# Séance 1 — Audit Qualité du Dataset MyAnimeList
**Projet : AniData Lab — Sakura Analytics**

Objectif : Explorer les fichiers bruts et identifier les problèmes de qualité.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

RAW_PATH = '../data/raw/'

## 1. Chargement des fichiers CSV

In [ ]:
anime_df = pd.read_csv(RAW_PATH + 'anime.csv')
ratings_df = pd.read_csv(RAW_PATH + 'rating_complete.csv')
synopsis_df = pd.read_csv(RAW_PATH + 'anime_with_synopsis.csv')

print(f'anime.csv       : {anime_df.shape[0]:,} lignes, {anime_df.shape[1]} colonnes')
print(f'rating_complete : {ratings_df.shape[0]:,} lignes, {ratings_df.shape[1]} colonnes')
print(f'with_synopsis   : {synopsis_df.shape[0]:,} lignes, {synopsis_df.shape[1]} colonnes')

## 2. Aperçu des données — anime.csv

In [ ]:
anime_df.head(5)

In [ ]:
anime_df.dtypes

In [ ]:
anime_df.describe(include='all')

## 3. Valeurs manquantes

In [ ]:
def audit_missing(df, name):
    missing = df.isnull().sum()
    pct = (missing / len(df) * 100).round(2)
    result = pd.DataFrame({'manquants': missing, 'pourcentage': pct})
    result = result[result['manquants'] > 0].sort_values('pourcentage', ascending=False)
    print(f'\n=== {name} ===')
    print(result if not result.empty else 'Aucune valeur manquante')

audit_missing(anime_df, 'anime.csv')
audit_missing(ratings_df, 'rating_complete.csv')
audit_missing(synopsis_df, 'anime_with_synopsis.csv')

## 4. Doublons

In [ ]:
print(f'Doublons anime.csv       : {anime_df.duplicated().sum()}')
print(f'Doublons rating_complete : {ratings_df.duplicated().sum()}')
print(f'Doublons synopsis        : {synopsis_df.duplicated().sum()}')

## 5. Types incohérents

In [ ]:
# Colonnes qui devraient être numériques
print('=== Score anime ===')
print(anime_df['Score'].value_counts().head(10))
print('\nValeurs non-numériques dans Score :')
non_num = anime_df[pd.to_numeric(anime_df['Score'], errors='coerce').isna()]['Score'].unique()
print(non_num)

In [ ]:
# Vérification des épisodes
print('Valeurs non-numériques dans Episodes :')
non_num_ep = anime_df[pd.to_numeric(anime_df['Episodes'], errors='coerce').isna()]['Episodes'].unique()
print(non_num_ep)

## 6. Encodages — Titres japonais

In [ ]:
# Titres avec caractères non-ASCII
has_japanese = anime_df['Name'].str.contains(r'[^\x00-\x7F]', na=False)
print(f'Titres avec caractères non-ASCII : {has_japanese.sum()}')
print('\nExemples :')
print(anime_df[has_japanese]['Name'].head(10).tolist())

## 7. Distribution des ratings

In [ ]:
print('Distribution des ratings :')
print(ratings_df['rating'].value_counts().sort_index())
print(f'\nRange : {ratings_df["rating"].min()} - {ratings_df["rating"].max()}')

## 8. Résumé — Rapport d'audit

In [ ]:
print('='*50)
print('RAPPORT D\'AUDIT — AniData Lab')
print('='*50)
print()
print('PROBLÈMES IDENTIFIÉS :')
print('1. Score et Episodes contiennent "Unknown" (str) → à convertir en NaN puis float/int')
print('2. Genres, Studios, Producers sont des listes en string → à parser')
print('3. Aired contient des dates en format texte → à normaliser')
print('4. Titres japonais/coréens en UTF-8 → à gérer lors de l\'indexation')
print('5. Ratings -1 dans certains datasets → valeur sentinelle à remplacer par NaN')
print()
print('ACTIONS PRÉVUES (séance 2) :')
print('- Remplacer "Unknown" par NaN')
print('- Convertir Score, Episodes en float/int')
print('- Parser les colonnes multi-valeurs (genres, studios)')
print('- Feature engineering : score_popularité, ratio_completion')
print('- Export dataset gold en CSV + JSON')